In [1]:
"""
BNPS Wisconsin Breast Cancer — Best Results Script
===================================================
Improvements over all previous notebooks:
  1. All 30 features (not just 10 mean features)
  2. StandardScaler normalization (prevents gradient explosion)
  3. Lower LR=0.05 with momentum (prevents divergence at high steps)
  4. Stratified membrane sampling (balanced class distribution)
  5. Piecewise sigmoid: 3-segment approx (better than 0.5+0.25z)
  6. Sweeps membranes AND steps to find global best
  7. Full benchmark vs sklearn LR, PyTorch SLP, PyTorch Deep MLP
  8. Wilson 95% CI on all accuracies
  9. Saves graphs: wisconsin_best_accuracy.png, wisconsin_best_speedup.png

Run on Google Colab (T4 GPU) or any machine with Python + sklearn + torch.
No CUDA kernel required — pure Python BNPS + PyTorch GPU baselines.
"""

import time, sys, warnings
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from statsmodels.stats.proportion import proportion_confint
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# ── CONFIG ────────────────────────────────────────────────────────────
MEMBRANE_COUNTS = [25, 50, 75, 100, 150, 200]
STEP_SWEEP      = [10, 25, 50, 75, 100, 150]
LR              = 0.05          # lower than original 0.5 → prevents divergence
MOMENTUM        = 0.85          # momentum on gradients
MLP_EPOCHS      = 100
RUNS            = 10            # timing repetitions
TEST_SIZE       = 0.20
RANDOM_STATE    = 42
# ──────────────────────────────────────────────────────────────────────

print("=" * 65)
print("  BNPS Wisconsin Breast Cancer — Best Results Benchmark")
print("=" * 65)

# ── 1. LOAD DATA (all 30 features) ───────────────────────────────────
data   = load_breast_cancer()
X_all  = data.data.astype('float32')      # (569, 30)
y_all  = data.target.astype('float32')    # 0=malignant, 1=benign

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE,
    stratify=y_all, random_state=RANDOM_STATE)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype('float32')
X_test  = scaler.transform(X_test_raw).astype('float32')

NF = X_train.shape[1]    # 30
N_TRAIN = len(X_train)
N_TEST  = len(X_test)

print(f"\n  Dataset : Breast Cancer Wisconsin (UCI)")
print(f"  Features: {NF} (all 30, StandardScaler normalized)")
print(f"  Train   : {N_TRAIN}  |  Test: {N_TEST}")
print(f"  Class balance (train): "
      f"benign={int(y_train.sum())}  malignant={int((y_train==0).sum())}")

# ── 2. HELPERS ────────────────────────────────────────────────────────
def wilson_ci(acc, n, alpha=0.05):
    lo, hi = proportion_confint(int(acc * n), n, alpha=alpha, method='wilson')
    return lo, hi

def compute_metrics(y_true, y_pred_prob, thresh=0.5):
    y_pred = (y_pred_prob >= thresh).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)
    lo, hi = wilson_ci(acc, len(y_true))
    return dict(acc=acc, prec=prec, rec=rec, f1=f1, cm=cm, ci=(lo, hi))

def piecewise_sigmoid(z):
    """
    3-piece piecewise sigmoid — better than 0.5+0.25z for large |z|.
      z < -2  ->  0.1*z + 0.5   (slope 0.1, clips near 0)
      -2<=z<=2 -> 0.25*z + 0.5  (slope 0.25, linear region)
      z > 2   ->  0.1*z + 0.5   (slope 0.1, clips near 1)
    Clipped to [0.01, 0.99].
    """
    sig = np.where(np.abs(z) <= 2,
                   0.25 * z + 0.5,
                   0.1  * z + 0.5)
    return np.clip(sig, 0.01, 0.99)

# ── 3. MEMBRANE SLP (BNPS Python simulation) ──────────────────────────
class MembraneSLP:
    """
    BNPS Single-Layer Perceptron simulator.
    Each training sample = one membrane; controller membrane holds weights.
    Uses piecewise sigmoid + SGD with momentum.
    """
    def __init__(self, X, y, lr=0.05, momentum=0.85):
        self.X  = X
        self.y  = y
        self.lr = lr
        self.mu = momentum
        self.F  = X.shape[1]
        self.w  = np.zeros(self.F, dtype='float64')
        self.b  = 0.0
        self.vw = np.zeros(self.F, dtype='float64')   # velocity (momentum)
        self.vb = 0.0
        self.loss_history = []

    def reset(self):
        self.w  = np.zeros(self.F, dtype='float64')
        self.b  = 0.0
        self.vw = np.zeros(self.F, dtype='float64')
        self.vb = 0.0
        self.loss_history = []

    def step(self):
        """One BNPS step: parallel forward + gradient aggregate + weight update."""
        z     = self.X @ self.w + self.b          # (N,) — all membranes parallel
        sigma = piecewise_sigmoid(z)
        error = sigma - self.y                     # (N,)

        # Gradients in each sample membrane
        grad_w = (error[:, None] * self.X).mean(axis=0)   # (F,)
        grad_b = error.mean()

        # BCE loss for monitoring
        loss = -np.mean(self.y * np.log(sigma) + (1 - self.y) * np.log(1 - sigma))
        self.loss_history.append(float(loss))

        # Controller: SGD with momentum
        self.vw = self.mu * self.vw + self.lr * grad_w
        self.vb = self.mu * self.vb + self.lr * grad_b
        self.w  -= self.vw
        self.b  -= self.vb

    def train(self, n_steps):
        self.reset()
        for _ in range(n_steps):
            self.step()

    def predict_prob(self, X):
        z = X @ self.w + self.b
        return piecewise_sigmoid(z)

    def predict(self, X, thresh=0.5):
        return (self.predict_prob(X) >= thresh).astype(int)

def stratified_sample(X, y, n):
    """Balanced class sampling: n//2 from each class."""
    n = min(n, len(X))
    n_pos = n // 2
    n_neg = n - n_pos
    rng  = np.random.default_rng(42)
    pos  = rng.choice(np.where(y == 1)[0], min(n_pos, (y==1).sum()), replace=False)
    neg  = rng.choice(np.where(y == 0)[0], min(n_neg, (y==0).sum()), replace=False)
    idx  = np.concatenate([pos, neg])
    rng.shuffle(idx)
    return X[idx], y[idx]

# ── 4. SKLEARN BASELINES ──────────────────────────────────────────────
print("\n[1] Baseline: sklearn Logistic Regression (full data)")
t0      = time.time()
lr_clf  = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_clf.fit(X_train, y_train)
lr_train_ms = (time.time() - t0) * 1000
y_prob_lr   = lr_clf.predict_proba(X_test)[:, 1]
lr_m        = compute_metrics(y_test, y_prob_lr)
lo, hi      = lr_m['ci']
print(f"  sklearn LR : Acc={lr_m['acc']:.4f}  F1={lr_m['f1']:.4f}  "
      f"CI=[{lo:.3f}-{hi:.3f}]  Train={lr_train_ms:.1f}ms")

# ── 5. PYTORCH BASELINES ──────────────────────────────────────────────
dev  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Xtr  = torch.tensor(X_train).to(dev)
ytr  = torch.tensor(y_train).to(dev)
Xte  = torch.tensor(X_test).to(dev)

print(f"\n[2] PyTorch baselines (device={dev})")

class LinearSLP(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.fc = nn.Linear(nf, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc(x)).squeeze(1)

class DeepMLP(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nf, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1),   nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

def train_pt(model, X, y, epochs, lr_pt=1e-3):
    opt  = torch.optim.Adam(model.parameters(), lr=lr_pt, weight_decay=1e-4)
    loss_fn = nn.BCELoss()
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(X), y).backward(); opt.step()
    if torch.cuda.is_available(): torch.cuda.synchronize()

def eval_pt(model, X, y_np):
    model.eval()
    with torch.no_grad():
        prob = model(X).cpu().numpy()
    return compute_metrics(y_np, prob)

# PyTorch SLP (1-layer, full data)
pt_slp_times = []
for _ in range(RUNS):
    m = LinearSLP(NF).to(dev)
    t0 = time.time()
    train_pt(m, Xtr, ytr, MLP_EPOCHS)
    pt_slp_times.append((time.time()-t0)*1000)
PT_SLP_MS = float(np.median(pt_slp_times))
pt_slp_m  = eval_pt(m, Xte, y_test)
lo, hi    = pt_slp_m['ci']
print(f"  PT SLP  (1-layer, {MLP_EPOCHS}ep)  : Acc={pt_slp_m['acc']:.4f}  "
      f"F1={pt_slp_m['f1']:.4f}  CI=[{lo:.3f}-{hi:.3f}]  Train={PT_SLP_MS:.1f}ms")

# PyTorch Deep MLP (4-layer, full data)
pt_mlp_times = []
for _ in range(RUNS):
    m = DeepMLP(NF).to(dev)
    t0 = time.time()
    train_pt(m, Xtr, ytr, MLP_EPOCHS)
    pt_mlp_times.append((time.time()-t0)*1000)
PT_MLP_MS = float(np.median(pt_mlp_times))
pt_mlp_m  = eval_pt(m, Xte, y_test)
lo, hi    = pt_mlp_m['ci']
print(f"  PT MLP (4-layer, {MLP_EPOCHS}ep)   : Acc={pt_mlp_m['acc']:.4f}  "
      f"F1={pt_mlp_m['f1']:.4f}  CI=[{lo:.3f}-{hi:.3f}]  Train={PT_MLP_MS:.1f}ms")

# ── 6. BNPS STEP SWEEP (fixed 100 membranes) ─────────────────────────
print("\n[3] BNPS Step Sweep (fixed 100 membranes, lr=0.05, momentum=0.85)")
print(f"  {'Steps':>6}  {'Acc':>8}  {'F1':>7}  {'Prec':>7}  {'Rec':>7}  {'95% CI':>18}")
print("  " + "-" * 60)

FIXED_MEMS = 100
Xm, ym     = stratified_sample(X_train, y_train, FIXED_MEMS)
slp_fixed  = MembraneSLP(Xm, ym, lr=LR, momentum=MOMENTUM)
step_results = []

for steps in STEP_SWEEP:
    slp_fixed.train(steps)
    prob = slp_fixed.predict_prob(X_test)
    sm   = compute_metrics(y_test, prob)
    lo, hi = sm['ci']
    step_results.append((steps, sm))
    print(f"  {steps:>6}  {sm['acc']:>8.4f}  {sm['f1']:>7.4f}  "
          f"{sm['prec']:>7.4f}  {sm['rec']:>7.4f}  [{lo:.3f}-{hi:.3f}]")

best_step_idx  = int(np.argmax([r['acc'] for _, r in step_results]))
best_steps     = step_results[best_step_idx][0]
best_step_acc  = step_results[best_step_idx][1]['acc']
print(f"\n  Best: {best_steps} steps → Acc={best_step_acc:.4f}")

# ── 7. BNPS MEMBRANE SWEEP (best steps) ───────────────────────────────
print(f"\n[4] BNPS Membrane Sweep (steps={best_steps}, lr={LR}, momentum={MOMENTUM})")
print(f"  {'Mems':>6}  {'Acc':>8}  {'F1':>7}  {'Prec':>7}  {'Rec':>7}  "
      f"{'Time(ms)':>10}  {'95% CI':>18}")
print("  " + "-" * 70)

mem_results = []
for nm in MEMBRANE_COUNTS:
    Xm, ym = stratified_sample(X_train, y_train, nm)
    slp    = MembraneSLP(Xm, ym, lr=LR, momentum=MOMENTUM)

    times  = []
    for _ in range(RUNS):
        t0 = time.time()
        slp.train(best_steps)
        times.append((time.time()-t0)*1000)
    train_ms = float(np.median(times))

    prob  = slp.predict_prob(X_test)
    mm    = compute_metrics(y_test, prob)
    lo, hi = mm['ci']
    mem_results.append((nm, mm, train_ms))
    print(f"  {nm:>6}  {mm['acc']:>8.4f}  {mm['f1']:>7.4f}  "
          f"{mm['prec']:>7.4f}  {mm['rec']:>7.4f}  "
          f"{train_ms:>10.1f}  [{lo:.3f}-{hi:.3f}]")

best_mem_idx  = int(np.argmax([r['acc'] for _, r, _ in mem_results]))
best_nm       = mem_results[best_mem_idx][0]
best_mem_m    = mem_results[best_mem_idx][1]
best_mem_ms   = mem_results[best_mem_idx][2]
print(f"\n  Best: {best_nm} membranes, {best_steps} steps → Acc={best_mem_m['acc']:.4f}")

# ── 8. FINAL SUMMARY ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  FINAL RESULTS — BNPS Wisconsin Breast Cancer")
print(f"{'='*65}")
print(f"\n  {'Method':<35}  {'Acc':>8}  {'F1':>7}  {'Train(ms)':>10}  {'95% CI'}")
print("  " + "-" * 72)

for name, m, ms in [
    ("sklearn LogReg (full data)",        lr_m,       lr_train_ms),
    (f"PT SLP 1-layer ({MLP_EPOCHS}ep)",  pt_slp_m,  PT_SLP_MS),
    (f"PT MLP 4-layer ({MLP_EPOCHS}ep)",  pt_mlp_m,  PT_MLP_MS),
    (f"BNPS ({best_nm}mem, {best_steps}steps) ← BEST", best_mem_m, best_mem_ms),
]:
    lo, hi = m['ci']
    print(f"  {name:<35}  {m['acc']:>8.4f}  {m['f1']:>7.4f}  "
          f"{ms:>10.1f}  [{lo:.3f}-{hi:.3f}]")

# Speed ratios
print(f"\n  BNPS vs PT SLP  : {PT_SLP_MS/best_mem_ms:.1f}x faster")
print(f"  BNPS vs PT MLP  : {PT_MLP_MS/best_mem_ms:.1f}x faster")
print(f"  BNPS Acc vs PT SLP : {(best_mem_m['acc']-pt_slp_m['acc'])*100:+.2f}%")
print(f"  BNPS Acc vs PT MLP : {(best_mem_m['acc']-pt_mlp_m['acc'])*100:+.2f}%")
print(f"\n  Config: LR={LR}  Momentum={MOMENTUM}  "
      f"Sigmoid=piecewise-3seg  Features=30/30")

# Confusion matrix for best BNPS
cm = best_mem_m['cm']
print(f"\n  BNPS Best Confusion Matrix ({best_nm}m, {best_steps}steps):")
print(f"    Predicted →     Benign  Malignant")
print(f"    Actual Benign    TN={cm[1][1]:3d}    FP={cm[1][0]:3d}")
print(f"    Actual Malignant FN={cm[0][1]:3d}    TP={cm[0][0]:3d}")
print(f"{'='*65}")

# ── 9. PLOTS ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("BNPS Wisconsin Breast Cancer — Best Results",
             fontsize=13, fontweight='bold')

# Plot 1: Accuracy vs Steps
ax = axes[0]
steps_x = [s for s, _ in step_results]
accs_s  = [r['acc'] for _, r in step_results]
f1s_s   = [r['f1']  for _, r in step_results]
ax.plot(steps_x, accs_s, 'o-', color='#3498db', lw=2.5, ms=8, label='Accuracy')
ax.plot(steps_x, f1s_s,  's--', color='#e74c3c', lw=2,   ms=7, label='F1 Score')
for x, v in zip(steps_x, accs_s):
    ax.annotate(f'{v:.3f}', (x, v), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=9)
ax.axvline(best_steps, color='#27ae60', ls=':', lw=1.5, label=f'Best={best_steps}')
ax.set_xlabel('BNPS Steps (iterations)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title(f'Acc vs Steps\n(100 membranes, LR={LR}, mom={MOMENTUM})', fontsize=10)
ax.set_ylim(0.75, 1.05); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Plot 2: Accuracy vs Membranes
ax = axes[1]
mems_x = [nm for nm, _, _ in mem_results]
accs_m  = [r['acc'] for _, r, _ in mem_results]
f1s_m   = [r['f1']  for _, r, _ in mem_results]
ax.plot(mems_x, accs_m, 'o-', color='#9b59b6', lw=2.5, ms=8, label='Accuracy')
ax.plot(mems_x, f1s_m,  's--', color='#e67e22', lw=2,   ms=7, label='F1 Score')
for x, v in zip(mems_x, accs_m):
    ax.annotate(f'{v:.3f}', (x, v), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=9)
ax.axhline(pt_mlp_m['acc'], color='#e74c3c', ls='--', lw=1.5,
           label=f"PT MLP={pt_mlp_m['acc']:.3f}")
ax.axhline(pt_slp_m['acc'], color='#3498db', ls='--', lw=1.5,
           label=f"PT SLP={pt_slp_m['acc']:.3f}")
ax.set_xlabel('BNPS Membrane Count', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title(f'Acc vs Membranes\n({best_steps} steps, LR={LR})', fontsize=10)
ax.set_ylim(0.75, 1.05); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Plot 3: Method comparison bar chart
ax = axes[2]
methods = ['LR\n(full)', f'PT SLP\n({MLP_EPOCHS}ep)', f'PT MLP\n({MLP_EPOCHS}ep)',
           f'BNPS\n({best_nm}m,{best_steps}s)']
accs_bar = [lr_m['acc'], pt_slp_m['acc'], pt_mlp_m['acc'], best_mem_m['acc']]
f1s_bar  = [lr_m['f1'],  pt_slp_m['f1'],  pt_mlp_m['f1'],  best_mem_m['f1']]
colors   = ['#95a5a6', '#3498db', '#e74c3c', '#27ae60']
x = np.arange(len(methods)); w = 0.35
b1 = ax.bar(x - w/2, accs_bar, w, label='Accuracy', color=colors, alpha=0.85)
b2 = ax.bar(x + w/2, f1s_bar,  w, label='F1 Score',  color=colors, alpha=0.5, hatch='//')
for b in list(b1) + list(b2):
    ax.annotate(f'{b.get_height():.3f}',
                xy=(b.get_x() + b.get_width()/2, b.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=10)
ax.set_ylim(0.75, 1.10); ax.set_ylabel('Score', fontsize=11)
ax.set_title('Method Comparison\n(Breast Cancer Wisconsin)', fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('wisconsin_best_results.png', dpi=150, bbox_inches='tight')
print("\nPlot saved: wisconsin_best_results.png")

# Loss curve
fig2, ax2 = plt.subplots(figsize=(10, 4))
slp_loss  = MembraneSLP(*stratified_sample(X_train, y_train, best_nm),
                         lr=LR, momentum=MOMENTUM)
slp_loss.train(max(STEP_SWEEP))
ax2.plot(range(1, len(slp_loss.loss_history)+1), slp_loss.loss_history,
         color='#2ecc71', lw=2.5, label=f'BNPS ({best_nm}m, LR={LR}, mom={MOMENTUM})')
ax2.set_xlabel('BNPS Step (Iteration)', fontsize=12)
ax2.set_ylabel('Binary Cross-Entropy Loss', fontsize=12)
ax2.set_title('BNPS Training Loss Curve — Breast Cancer Wisconsin', fontsize=13)
ax2.legend(fontsize=11); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('wisconsin_best_loss.png', dpi=150, bbox_inches='tight')
print("Plot saved: wisconsin_best_loss.png")


  BNPS Wisconsin Breast Cancer — Best Results Benchmark

  Dataset : Breast Cancer Wisconsin (UCI)
  Features: 30 (all 30, StandardScaler normalized)
  Train   : 455  |  Test: 114
  Class balance (train): benign=285  malignant=170

[1] Baseline: sklearn Logistic Regression (full data)
  sklearn LR : Acc=0.9825  F1=0.9861  CI=[0.938-0.995]  Train=20.9ms

[2] PyTorch baselines (device=cuda)
  PT SLP  (1-layer, 100ep)  : Acc=0.9298  F1=0.9429  CI=[0.868-0.964]  Train=101.5ms
  PT MLP (4-layer, 100ep)   : Acc=0.9561  F1=0.9645  CI=[0.901-0.981]  Train=197.7ms

[3] BNPS Step Sweep (fixed 100 membranes, lr=0.05, momentum=0.85)
   Steps       Acc       F1     Prec      Rec              95% CI
  ------------------------------------------------------------
      10    0.9035   0.9209   0.9552   0.8889  [0.835-0.945]
      25    0.9561   0.9650   0.9718   0.9583  [0.901-0.981]
      50    0.9649   0.9718   0.9857   0.9583  [0.913-0.986]
      75    0.9737   0.9790   0.9859   0.9722  [0.925-0.991

In [9]:
# ═══════════════════════════════════════════════════════════════
# wisc_baselines.py  —  Self-contained: PT SLP/MLP + TF SLP/MLP
# Paste as ANY cell in Colab — defines everything it needs.
# Outputs: wisc_final_comparison.png, wisc_dl_results.json
# ═══════════════════════════════════════════════════════════════
import time, json, os, warnings
import numpy as np
import torch, torch.nn as nn
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ── CONFIG (edit here if needed) ──────────────────────────────
MEMBRANE_COUNTS = [25, 50, 100, 150, 200]
STEPS_SWEEP     = [10, 25, 50, 100]
RUNS            = 10
TEST_SIZE       = 0.20
RANDOM_STATE    = 42

# ── DATA ──────────────────────────────────────────────────────
_d = load_breast_cancer()
_X, _y = _d.data.astype('float32'), _d.target.astype('float32')
X_train, X_test, y_train, y_test = train_test_split(
    _X, _y, test_size=TEST_SIZE, stratify=_y, random_state=RANDOM_STATE)
_sc = StandardScaler()
X_train = _sc.fit_transform(X_train).astype('float32')
X_test  = _sc.transform(X_test).astype('float32')
NF = X_train.shape[1]   # 30

# ── HELPERS ───────────────────────────────────────────────────
def stratified_sample(X, y, n, seed=42):
    n   = min(n, len(X))
    rng = np.random.default_rng(seed)
    pos = rng.choice(np.where(y==1)[0], n//2, replace=False)
    neg = rng.choice(np.where(y==0)[0], n-n//2, replace=False)
    idx = np.concatenate([pos, neg]); rng.shuffle(idx)
    return X[idx], y[idx]

def metrics(y_true, y_pred_prob, thresh=0.5):
    yp  = (y_pred_prob >= thresh).astype(int)
    acc = accuracy_score(y_true, yp)
    return dict(
        acc  = acc,
        prec = precision_score(y_true, yp, zero_division=0),
        rec  = recall_score(y_true, yp, zero_division=0),
        f1   = f1_score(y_true, yp, zero_division=0),
    )

# ── DEVICE ────────────────────────────────────────────────────

# Optional TensorFlow import
try:
    import tensorflow as tf
    tf.get_logger().setLevel('ERROR')
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    TF_OK = True
except ImportError:
    TF_OK = False
    print("TensorFlow not available — TF baselines skipped")

dev  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Xte  = torch.tensor(X_test).to(dev)
print(f"Device: {dev}  |  TF available: {TF_OK}\n")

# ── PyTorch models ────────────────────────────────────────────────────
class PT_SLP(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.fc = nn.Linear(nf, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc(x)).squeeze(1)

class PT_MLP(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nf, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),  nn.Sigmoid())
    def forward(self, x):
        return self.net(x).squeeze(1)

def train_pt(model, Xd, yd, epochs):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    fn  = nn.BCELoss()
    t0  = time.time()
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        fn(model(Xd), yd).backward(); opt.step()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return (time.time()-t0)*1000

def eval_pt(model):
    model.eval()
    with torch.no_grad():
        prob = model(Xte).cpu().numpy()
    return metrics(y_test, prob)

# ── TF models ─────────────────────────────────────────────────────────
def make_tf_slp(nf):
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(1, activation='sigmoid', input_shape=(nf,))])
    m.compile(optimizer='adam', loss='binary_crossentropy')
    return m

def make_tf_mlp(nf):
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(nf,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid')])
    m.compile(optimizer='adam', loss='binary_crossentropy')
    return m

def train_tf(model, Xd, yd, epochs):
    t0 = time.time()
    model.fit(Xd, yd, epochs=epochs, batch_size=len(Xd), verbose=0)
    return (time.time()-t0)*1000

def eval_tf(model):
    prob = model.predict(X_test, verbose=0).flatten()
    return metrics(y_test, prob)

# ── SWEEP ─────────────────────────────────────────────────────────────
dl_results = {}

print(f"{'Mems':>5} {'Eps':>5} | "
      f"{'PT-SLP':>9} {'ms':>7} | "
      f"{'PT-MLP':>9} {'ms':>7} | "
      f"{'TF-SLP':>9} {'ms':>7} | "
      f"{'TF-MLP':>9} {'ms':>7}")
print("-" * 80)

for nm in MEMBRANE_COUNTS:
    Xm, ym = stratified_sample(X_train, y_train, nm)
    Xm_t   = torch.tensor(Xm).to(dev)
    ym_t   = torch.tensor(ym).to(dev)

    for epochs in STEPS_SWEEP:
        key = f"{nm}_{epochs}"
        row = dict(nm=nm, epochs=epochs)

        # ── PyTorch SLP ──────────────────────────────────────────────
        pt_slp_ms_list = []
        for _ in range(RUNS):
            m = PT_SLP(NF).to(dev)
            pt_slp_ms_list.append(train_pt(m, Xm_t, ym_t, epochs))
        row['pt_slp_ms'] = float(np.median(pt_slp_ms_list))
        row.update({f'pt_slp_{k}': v for k, v in eval_pt(m).items()})

        # ── PyTorch MLP ──────────────────────────────────────────────
        pt_mlp_ms_list = []
        for _ in range(RUNS):
            m = PT_MLP(NF).to(dev)
            pt_mlp_ms_list.append(train_pt(m, Xm_t, ym_t, epochs))
        row['pt_mlp_ms'] = float(np.median(pt_mlp_ms_list))
        row.update({f'pt_mlp_{k}': v for k, v in eval_pt(m).items()})

        # ── TF SLP ───────────────────────────────────────────────────
        if TF_OK:
            tf_slp_ms_list = []
            for _ in range(RUNS):
                m = make_tf_slp(NF)
                tf_slp_ms_list.append(train_tf(m, Xm, ym, epochs))
            row['tf_slp_ms'] = float(np.median(tf_slp_ms_list))
            tm = eval_tf(m)
            row.update({f'tf_slp_{k}': v for k, v in tm.items()})

            # ── TF MLP ───────────────────────────────────────────────
            tf_mlp_ms_list = []
            for _ in range(RUNS):
                m = make_tf_mlp(NF)
                tf_mlp_ms_list.append(train_tf(m, Xm, ym, epochs))
            row['tf_mlp_ms'] = float(np.median(tf_mlp_ms_list))
            tm = eval_tf(m)
            row.update({f'tf_mlp_{k}': v for k, v in tm.items()})

        dl_results[key] = row

        tf_s = f"{row.get('tf_slp_acc', float('nan')):.4f}/{row.get('tf_slp_ms', 0):.0f}ms" if TF_OK else "N/A"
        tf_m = f"{row.get('tf_mlp_acc', float('nan')):.4f}/{row.get('tf_mlp_ms', 0):.0f}ms" if TF_OK else "N/A"
        print(f"{nm:>5} {epochs:>5} | "
              f"{row['pt_slp_acc']:.4f} {row['pt_slp_ms']:>6.0f}ms | "
              f"{row['pt_mlp_acc']:.4f} {row['pt_mlp_ms']:>6.0f}ms | "
              f"{tf_s:>17} | {tf_m:>17}")

with open('wisc_dl_results.json', 'w') as f:
    json.dump(dl_results, f, indent=2)
print("\nSaved: wisc_dl_results.json")

# ── LOAD BNPS RESULTS ─────────────────────────────────────────────────
bnps_results = {}
if os.path.exists('wisc_bnps_results.json'):
    with open('wisc_bnps_results.json') as f:
        bnps_results = json.load(f)
else:
    print("wisc_bnps_results.json not found — run wisc_bnps.py first")

# ── PLOTS ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('BNPS vs PyTorch vs TensorFlow — Wisconsin Breast Cancer\n'
             'Fair Comparison: Same N Samples, Same Steps/Epochs',
             fontsize=13, fontweight='bold')

COLORS = {
    'BNPS Serial': '#27ae60',
    'BNPS CUDA':   '#2ecc71',
    'PT SLP':      '#3498db',
    'PT MLP':      '#e74c3c',
    'TF SLP':      '#9b59b6',
    'TF MLP':      '#e67e22',
}

# ── Plot 1: Acc vs Membranes (fixed steps = max STEPS_SWEEP) ──────────
ax = axes[0, 0]
fix_steps = max(STEPS_SWEEP)
for label, src, acc_key, ms_key in [
    ('BNPS Serial', bnps_results, 'acc',        'serial_ms'),
    ('BNPS CUDA',   bnps_results, 'acc',        'cuda_ms'),
    ('PT SLP',      dl_results,   'pt_slp_acc', 'pt_slp_ms'),
    ('PT MLP',      dl_results,   'pt_mlp_acc', 'pt_mlp_ms'),
]:
    xs, ys = [], []
    for nm in MEMBRANE_COUNTS:
        k = f"{nm}_{fix_steps}"
        if k in src and not np.isnan(src[k].get(acc_key, float('nan'))):
            xs.append(nm); ys.append(src[k][acc_key])
    if xs:
        ax.plot(xs, ys, 'o-', color=COLORS[label], lw=2, ms=7, label=label)

if TF_OK:
    for label, acc_key in [('TF SLP','tf_slp_acc'),('TF MLP','tf_mlp_acc')]:
        xs, ys = [], []
        for nm in MEMBRANE_COUNTS:
            k = f"{nm}_{fix_steps}"
            if k in dl_results: xs.append(nm); ys.append(dl_results[k].get(acc_key, float('nan')))
        if xs: ax.plot(xs, ys, 's--', color=COLORS[label], lw=2, ms=7, label=label)

ax.set_xlabel('Membrane Count (= Training Samples)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title(f'Accuracy vs Membrane Count\n(steps/epochs = {fix_steps})', fontsize=10)
ax.set_ylim(0.70, 1.05); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── Plot 2: Acc vs Steps (fixed nm = median membrane count) ───────────
ax = axes[0, 1]
fix_nm = MEMBRANE_COUNTS[len(MEMBRANE_COUNTS)//2]
for label, src, acc_key in [
    ('BNPS Serial', bnps_results, 'acc'),
    ('PT SLP',      dl_results,   'pt_slp_acc'),
    ('PT MLP',      dl_results,   'pt_mlp_acc'),
]:
    xs, ys = [], []
    for steps in STEPS_SWEEP:
        k = f"{fix_nm}_{steps}"
        if k in src and not np.isnan(src[k].get(acc_key, float('nan'))):
            xs.append(steps); ys.append(src[k][acc_key])
    if xs: ax.plot(xs, ys, 'o-', color=COLORS[label], lw=2, ms=7, label=label)

if TF_OK:
    for label, acc_key in [('TF SLP','tf_slp_acc'),('TF MLP','tf_mlp_acc')]:
        xs, ys = [], []
        for steps in STEPS_SWEEP:
            k = f"{fix_nm}_{steps}"
            if k in dl_results: xs.append(steps); ys.append(dl_results[k].get(acc_key, float('nan')))
        if xs: ax.plot(xs, ys, 's--', color=COLORS[label], lw=2, ms=7, label=label)

ax.set_xlabel('Steps / Epochs', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title(f'Accuracy vs Steps/Epochs\n(membranes = {fix_nm})', fontsize=10)
ax.set_ylim(0.70, 1.05); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── Plot 3: Training Time vs Membranes ────────────────────────────────
ax = axes[1, 0]
for label, src, ms_key in [
    ('BNPS Serial', bnps_results, 'serial_ms'),
    ('BNPS CUDA',   bnps_results, 'cuda_ms'),
    ('PT SLP',      dl_results,   'pt_slp_ms'),
    ('PT MLP',      dl_results,   'pt_mlp_ms'),
]:
    xs, ys = [], []
    for nm in MEMBRANE_COUNTS:
        k = f"{nm}_{fix_steps}"
        if k in src and not np.isnan(src[k].get(ms_key, float('nan'))):
            xs.append(nm); ys.append(src[k][ms_key])
    if xs: ax.plot(xs, ys, 'o-', color=COLORS[label], lw=2, ms=7, label=label)

if TF_OK:
    for label, ms_key in [('TF SLP','tf_slp_ms'),('TF MLP','tf_mlp_ms')]:
        xs, ys = [], []
        for nm in MEMBRANE_COUNTS:
            k = f"{nm}_{fix_steps}"
            if k in dl_results: xs.append(nm); ys.append(dl_results[k].get(ms_key, 0))
        if xs: ax.plot(xs, ys, 's--', color=COLORS[label], lw=2, ms=7, label=label)

ax.set_xlabel('Membrane Count (= Training Samples)', fontsize=11)
ax.set_ylabel('Training Time (ms)', fontsize=11)
ax.set_title(f'Training Time vs Membrane Count\n(steps/epochs = {fix_steps})', fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── Plot 4: Best result bar chart (best cell per method) ──────────────
ax = axes[1, 1]
method_best = {}
for label, src, acc_key, ms_key in [
    ('BNPS\nSerial',  bnps_results, 'acc',        'serial_ms'),
    ('BNPS\nCUDA',    bnps_results, 'acc',        'cuda_ms'),
    ('PT\nSLP',       dl_results,   'pt_slp_acc', 'pt_slp_ms'),
    ('PT\nMLP',       dl_results,   'pt_mlp_acc', 'pt_mlp_ms'),
]:
    best_acc, best_ms = float('-inf'), 0
    for k, v in src.items():
        a = v.get(acc_key, float('nan'))
        if not np.isnan(a) and a > best_acc:
            best_acc = a
            best_ms  = v.get(ms_key, 0) or 0
    if best_acc > float('-inf'):
        method_best[label] = (best_acc, best_ms)

if TF_OK:
    for label, acc_key, ms_key in [
        ('TF\nSLP','tf_slp_acc','tf_slp_ms'),
        ('TF\nMLP','tf_mlp_acc','tf_mlp_ms')
    ]:
        best_acc, best_ms = float('-inf'), 0
        for k, v in dl_results.items():
            a = v.get(acc_key, float('nan'))
            if not np.isnan(a) and a > best_acc:
                best_acc = a; best_ms = v.get(ms_key, 0)
        if best_acc > float('-inf'):
            method_best[label] = (best_acc, best_ms)

labels   = list(method_best.keys())
accs     = [method_best[l][0] for l in labels]
col_list = [COLORS.get(l.replace('\n', ' '), '#7f8c8d') for l in labels]
bars = ax.bar(labels, accs, color=col_list, alpha=0.85, edgecolor='white')
for b in bars:
    ax.annotate(f'{b.get_height():.4f}',
                xy=(b.get_x()+b.get_width()/2, b.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0.75, 1.10)
ax.set_ylabel('Best Accuracy', fontsize=11)
ax.set_title('Best Accuracy per Method\n(over all membrane/step combinations)', fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('wisc_final_comparison.png', dpi=150, bbox_inches='tight')
print("Saved: wisc_final_comparison.png")

# ── TEXT SUMMARY ──────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  FINAL COMPARISON — Best per Method")
print(f"{'='*65}")
print(f"  {'Method':<20} {'Best Acc':>10} {'Train(ms)':>12}")
print("  " + "-"*44)
for label in labels:
    a, ms = method_best[label]
    print(f"  {label.replace(chr(10),' '):<20} {a:>10.4f} {ms:>12.1f}")
print(f"{'='*65}")


Device: cuda  |  TF available: True

 Mems   Eps |    PT-SLP      ms |    PT-MLP      ms |    TF-SLP      ms |    TF-MLP      ms
--------------------------------------------------------------------------------
   25    10 | 0.5439     19ms | 0.8509     55ms |     0.8421/1131ms |     0.8860/1807ms
   25    25 | 0.8596     26ms | 0.9123     39ms |     0.2368/1783ms |     0.9298/2413ms
   25    50 | 0.7982     52ms | 0.9474     78ms |     0.3158/2775ms |     0.9474/3475ms
   25   100 | 0.9123    105ms | 0.9386    157ms |     0.7105/5406ms |     0.9211/5849ms
   50    10 | 0.5702     12ms | 0.9561     16ms |     0.2895/1150ms |     0.8772/1848ms
   50    25 | 0.8421     27ms | 0.8947     40ms |     0.1754/1853ms |     0.9211/2443ms
   50    50 | 0.7368     50ms | 0.9123     77ms |     0.9123/2904ms |     0.9123/3651ms
   50   100 | 0.8860    105ms | 0.9123    169ms |     0.7368/4954ms |     0.8860/5854ms
  100    10 | 0.4649     11ms | 0.9035     16ms |     0.2368/1198ms |     0.8246/1874m

In [5]:
"""
wisc_bnps.py  —  BNPS Serial SLP + BNPS CUDA SLP benchmark.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Paste as Cell 2 in Colab AFTER running Cell 1 (wisc_config).
Also upload to Colab: bnps3.py, bnps_fast.cu (or bnps.cu)

Outputs: wisc_bnps_results.json
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# NOTE: No imports needed — Cell 1 (wisc_config) defines everything.
"""

import os, sys, re, time, json, subprocess, shutil
import numpy as np
# All other names (X_train, X_test, y_train, y_test, NF, MEMBRANE_COUNTS,
# STEPS_SWEEP, RUNS, stratified_sample, metrics, piecewise_sigmoid,
# MembraneSLP, make_pep, extract_weights, BNPS_LR) come from Cell 1.

CWD = os.getcwd()

# ── COMPILE CUDA ──────────────────────────────────────────────────────
print("Compiling BNPS CUDA kernel...")
_nvcc = shutil.which('nvcc')
if _nvcc is None:
    for p in ['/usr/local/cuda/bin/nvcc','/usr/local/cuda-12/bin/nvcc','/usr/bin/nvcc']:
        if os.path.exists(p): _nvcc = p; break

CUDA_OK = False
if _nvcc:
    cu = 'bnps_fast.cu' if os.path.exists('bnps_fast.cu') else 'bnps.cu'
    if os.path.exists(cu):
        ret = subprocess.run([_nvcc, '-O2', '-o', 'bnps_cuda', cu], capture_output=True)
        CUDA_OK = (ret.returncode == 0)
        print(f"  CUDA compiled OK ({cu})" if CUDA_OK else f"  CUDA compile FAILED: {ret.stderr.decode()[:200]}")
    else:
        print(f"  No .cu file found — CUDA disabled")
else:
    print("  nvcc not found — CUDA disabled (serial only)")

# ── STARTUP OVERHEAD ──────────────────────────────────────────────────
_oh = []
for _ in range(10):
    t = time.time()
    subprocess.run([sys.executable, 'bnps3.py'], capture_output=True, timeout=15)
    _oh.append((time.time()-t)*1000)
PY_OH = float(np.median(_oh))
print(f"  bnps3.py startup overhead: {PY_OH:.1f} ms\n")

# ── MAIN SWEEP ────────────────────────────────────────────────────────
results = {}   # key: (nm, steps)

print(f"{'Mems':>5} {'Steps':>6} {'Serial(ms)':>12} {'CUDA(ms)':>10} "
      f"{'Speedup':>9} {'Acc':>7} {'F1':>7}")
print("-" * 60)

for nm in MEMBRANE_COUNTS:
    Xm, ym = stratified_sample(X_train, y_train, nm)
    _idx   = slice(None, nm)

    for steps in STEPS_SWEEP:
        key = f"{nm}_{steps}"

        # ── Generate .pep file ────────────────────────────────────────
        pf = os.path.join(CWD, f'bc_{nm}_{steps}.pep')
        make_pep(Xm, ym, nm, NF, pf, lr=BNPS_LR)

        # ── BNPS Serial ───────────────────────────────────────────────
        ser_ts = []; last_out = ''
        for _ in range(RUNS):
            t0 = time.time()
            r  = subprocess.run([sys.executable, 'bnps3.py', pf, '-n', str(steps)],
                                capture_output=True, timeout=600, cwd=CWD)
            ser_ts.append(max((time.time()-t0)*1000 - PY_OH, 1.0))
            last_out = r.stdout.decode(errors='replace')

        ser_ms  = float(np.median(ser_ts))
        ser_std = float(np.std(ser_ts))

        # Extract weights → accuracy
        w, b = extract_weights(last_out, NF)
        acc_s = f1_s = prec_s = rec_s = float('nan')
        if w is not None:
            logits = X_test @ w + b
            prob   = 1 / (1 + np.exp(-logits))
            m_s    = metrics(y_test, prob)
            acc_s, f1_s = m_s['acc'], m_s['f1']
            prec_s, rec_s = m_s['prec'], m_s['rec']

        # ── BNPS CUDA ─────────────────────────────────────────────────
        cuda_ms = cuda_std = float('nan')
        if CUDA_OK:
            inp = os.path.join(CWD, 'input.txt')
            # Generate input.txt via -p flag
            subprocess.run([sys.executable, 'bnps3.py', pf, '-p', str(steps)],
                           capture_output=True, timeout=600, cwd=CWD)
            # Warmup
            subprocess.run(['./bnps_cuda', inp], capture_output=True, timeout=60, cwd=CWD)
            cuda_ts = []
            for _ in range(RUNS):
                t0  = time.time()
                rc  = subprocess.run(['./bnps_cuda', inp], capture_output=True,
                                     timeout=300, cwd=CWD)
                wall = (time.time()-t0)*1000
                out  = rc.stdout.decode(errors='replace')
                mx   = re.search(r'[Tt]ime[^:]*:\s*([\d.]+)\s*(ms|us|s)?', out)
                if mx:
                    v = float(mx.group(1)); u = (mx.group(2) or 'ms').lower()
                    cuda_ts.append(v*1e-3 if u=='us' else (v*1e3 if u=='s' else v))
                else:
                    cuda_ts.append(wall)
            cuda_ms  = float(np.median(cuda_ts))
            cuda_std = float(np.std(cuda_ts))

        speedup = ser_ms / cuda_ms if (CUDA_OK and cuda_ms > 0) else float('nan')

        results[key] = dict(
            nm=nm, steps=steps,
            serial_ms=ser_ms, serial_std=ser_std,
            cuda_ms=cuda_ms, cuda_std=cuda_std,
            speedup=speedup,
            acc=acc_s, f1=f1_s, prec=prec_s, rec=rec_s
        )

        acc_str = f"{acc_s:.4f}" if not np.isnan(acc_s) else "N/A"
        f1_str  = f"{f1_s:.4f}"  if not np.isnan(f1_s)  else "N/A"
        spd_str = f"{speedup:.2f}x" if not np.isnan(speedup) else "N/A"
        cud_str = f"{cuda_ms:.1f}" if not np.isnan(cuda_ms) else "N/A"
        print(f"{nm:>5} {steps:>6} {ser_ms:>12.1f} {cud_str:>10} "
              f"{spd_str:>9} {acc_str:>7} {f1_str:>7}")

# ── SAVE ─────────────────────────────────────────────────────────────
with open('wisc_bnps_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved: wisc_bnps_results.json")

# ── QUICK SUMMARY ─────────────────────────────────────────────────────
valid = [(v['acc'], k) for k, v in results.items() if not np.isnan(v['acc'])]
if valid:
    best_acc, best_key = max(valid)
    bv = results[best_key]
    print(f"\nBest BNPS Serial: nm={bv['nm']} steps={bv['steps']} "
          f"Acc={bv['acc']:.4f} F1={bv['f1']:.4f} Time={bv['serial_ms']:.1f}ms")
    if CUDA_OK and not np.isnan(bv['cuda_ms']):
        print(f"   CUDA: {bv['cuda_ms']:.1f}ms  Speedup={bv['speedup']:.2f}x")


Compiling BNPS CUDA kernel...
  CUDA compiled OK (bnps_fast.cu)
  bnps3.py startup overhead: 73.7 ms

 Mems  Steps   Serial(ms)   CUDA(ms)   Speedup     Acc      F1
------------------------------------------------------------
   25     10       3226.7       10.8   298.54x  0.9123  0.9296
   25     25       2905.1       22.7   128.09x  0.8860  0.9139
   25     50       2904.0       43.0    67.48x  0.9035  0.9281
   25    100       3175.8       84.0    37.81x  0.8860  0.9172
   50     10      11640.8       11.8   989.52x  0.9123  0.9306
   50     25      11661.1       26.5   439.91x  0.9035  0.9262
   50     50      11897.8       50.0   238.16x  0.9298  0.9452
   50    100      12191.7       97.7   124.79x  0.3947  0.5605
  100     10      43658.0       27.1  1609.84x  0.9123  0.9306


KeyboardInterrupt: 

In [6]:
"""
wisc_config.py  —  Shared config, data, helpers for Wisconsin benchmark.
Import this in wisc_bnps.py and wisc_baselines.py.

Sweep parameters
----------------
MEMBRANE_COUNTS : [25, 50, 100, 150, 200]
    Number of BNPS membranes = number of training samples used.
    All baselines (PT/TF) are ALSO trained on the SAME N samples
    so the comparison is perfectly fair.

STEPS_SWEEP : [10, 25, 50, 100]
    BNPS steps ≡ training epochs for baselines.
    Every method uses the same number of update iterations.

Dataset
-------
Breast Cancer Wisconsin (UCI), 569 samples, 30 features.
80/20 stratified split → 455 train / 114 test.
StandardScaler fit on train only.
"""

import numpy as np
import os, re, subprocess, sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from statsmodels.stats.proportion import proportion_confint

# ── SWEEP CONFIG ──────────────────────────────────────────────────────
MEMBRANE_COUNTS = [25, 50, 100, 150, 200]   # also = N training samples per run
STEPS_SWEEP     = [10, 25, 50, 100]          # BNPS steps = DL epochs
RUNS            = 10                         # timing repetitions (median taken)
TEST_SIZE       = 0.20
RANDOM_STATE    = 42

# BNPS hyper-params
BNPS_LR       = 0.05
BNPS_MOMENTUM = 0.85

# ── DATA ──────────────────────────────────────────────────────────────
def load_data():
    data  = load_breast_cancer()
    X_all = data.data.astype('float32')   # (569, 30)
    y_all = data.target.astype('float32') # 1=benign, 0=malignant

    X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
        X_all, y_all,
        test_size=TEST_SIZE, stratify=y_all,
        random_state=RANDOM_STATE)

    sc      = StandardScaler()
    X_train = sc.fit_transform(X_tr_raw).astype('float32')
    X_test  = sc.transform(X_te_raw).astype('float32')
    return X_train, X_test, y_train, y_test, sc

X_train, X_test, y_train, y_test, scaler = load_data()
NF       = X_train.shape[1]   # 30
N_TRAIN  = len(X_train)       # 455
N_TEST   = len(X_test)        # 114

# ── SAMPLING ──────────────────────────────────────────────────────────
def stratified_sample(X, y, n, seed=42):
    """Balanced class sample of size n from (X, y)."""
    n   = min(n, len(X))
    rng = np.random.default_rng(seed)
    pos = rng.choice(np.where(y == 1)[0], n // 2, replace=False)
    neg = rng.choice(np.where(y == 0)[0], n - n // 2, replace=False)
    idx = np.concatenate([pos, neg])
    rng.shuffle(idx)
    return X[idx], y[idx]

# ── METRICS ───────────────────────────────────────────────────────────
def metrics(y_true, y_pred_prob, thresh=0.5):
    yp  = (y_pred_prob >= thresh).astype(int)
    acc = accuracy_score(y_true, yp)
    lo, hi = proportion_confint(int(acc * len(y_true)), len(y_true),
                                alpha=0.05, method='wilson')
    return dict(
        acc  = acc,
        prec = precision_score(y_true, yp, zero_division=0),
        rec  = recall_score(y_true, yp, zero_division=0),
        f1   = f1_score(y_true, yp, zero_division=0),
        cm   = confusion_matrix(y_true, yp).tolist(),
        ci   = (lo, hi)
    )

# ── BNPS SIGMOID ──────────────────────────────────────────────────────
def piecewise_sigmoid(z):
    """3-segment piecewise sigmoid. Better than 0.5+0.25z for large |z|."""
    sig = np.where(np.abs(z) <= 2, 0.25*z + 0.5, 0.1*z + 0.5)
    return np.clip(sig, 0.01, 0.99)

# ── BNPS SERIAL SLP ───────────────────────────────────────────────────
class MembraneSLP:
    """
    Python simulation of BNPS SLP.
    Each sample membrane does forward + gradient in parallel.
    Controller membrane aggregates and updates weights.
    Uses piecewise sigmoid + SGD with momentum.
    """
    def __init__(self, X, y, lr=BNPS_LR, momentum=BNPS_MOMENTUM):
        self.X  = X.astype('float64')
        self.y  = y.astype('float64')
        self.lr = lr
        self.mu = momentum
        self.F  = X.shape[1]
        self.reset()

    def reset(self):
        self.w  = np.zeros(self.F)
        self.b  = 0.0
        self.vw = np.zeros(self.F)
        self.vb = 0.0
        self.loss_history = []

    def step(self):
        z     = self.X @ self.w + self.b
        sigma = piecewise_sigmoid(z)
        error = sigma - self.y
        grad_w = (error[:, None] * self.X).mean(axis=0)
        grad_b = error.mean()
        loss   = -np.mean(self.y*np.log(sigma) + (1-self.y)*np.log(1-sigma))
        self.loss_history.append(float(loss))
        self.vw = self.mu * self.vw + self.lr * grad_w
        self.vb = self.mu * self.vb + self.lr * grad_b
        self.w -= self.vw
        self.b -= self.vb

    def train(self, steps):
        self.reset()
        for _ in range(steps):
            self.step()

    def predict_prob(self, X):
        return piecewise_sigmoid(X.astype('float64') @ self.w + self.b)

# ── MAKE PEP (for BNPS CUDA) ──────────────────────────────────────────
def make_pep(Xd, yd, N, nf, fname, lr=0.01):
    """
    Generate BNPS .pep file for logistic SLP.
    Same format as BNPS_Cloud_Workload.py but for classification.
    """
    N  = min(N, len(Xd))
    Xs = Xd[:N].astype(float)
    ys = yd[:N].astype(float)
    f  = lambda v: f'{float(v):.6f}'
    ids = list(range(1, N + 2))

    L = ['bnps = {', '',
         f"    H = {{{','.join(str(i) for i in ids)}}};",
         f"    structure = [{' '.join(f'[ {i}' for i in ids)} "
         f"{' '.join(f']{i}' for i in reversed(ids))}];", '']

    cv = (','.join(f'w{i}' for i in range(nf)) + ',b,' +
          ','.join(f'tw{i}' for i in range(nf)) + ',tb,' +
          ','.join(f'lw{i}' for i in range(nf)) + ',lb,' +
          ','.join(f'aw{i}' for i in range(nf)) + ',ab')
    L += ['    1 = {', '        var = {', f'            {cv}', '        };', '']

    mul = N + 1
    for i in range(nf):
        targets = '+'.join(f'1|w{i}_{m}' for m in range(2, N+2)) + f'+1|tw{i}'
        L.append(f'        pr = {{ w{i}*{mul} -> {targets} }};')
    tb = '+'.join(f'1|b_{m}' for m in range(2, N+2)) + '+1|tb'
    L.append(f'        pr = {{ b*{mul} -> {tb} }};')
    L.append('')
    for i in range(nf):
        L.append(f'        pr = {{ {f(lr)} -> 1|lw{i} }};')
    L.append(f'        pr = {{ {f(lr)} -> 1|lb }};')
    L.append('')
    for i in range(nf):
        L.append(f'        pr = {{ {"+".join(f"gw{i}_{m}" for m in range(2,N+2))} -> 1|aw{i} }};')
    L.append(f'        pr = {{ {"+".join(f"gb_{m}" for m in range(2,N+2))} -> 1|ab }};')
    L.append('')
    for i in range(nf):
        L.append(f'        pr = {{ tw{i}-lw{i}*(aw{i}/{N}) -> 1|w{i} }};')
    L.append(f'        pr = {{ tb-lb*(ab/{N}) -> 1|b }};')
    L.append('')
    _init = [0.0] * ((nf + 1) * 4)
    L.append(f"        var0 = ({','.join(f'{v:.6f}' for v in _init)});")
    L += ['    };', '']

    for idx in range(N):
        m  = idx + 2
        xi = Xs[idx]; yi = float(ys[idx])
        sv = (','.join(f'w{i}_{m}' for i in range(nf)) + f',b_{m},z_{m},' +
              ','.join(f'za{j}_{m}' for j in range(nf+1)) + ',' +
              ','.join(f'gw{i}_{m}' for i in range(nf)) + f',gb_{m}')
        L += [f'    {m} = {{', '        var = {', f'            {sv}', '        };', '']
        fwd = '+'.join(f'w{i}_{m}*{f(xi[i])}' for i in range(nf)) + f'+b_{m}'
        L.append(f'        pr = {{ {fwd} -> 1|z_{m} }};')
        za_t = '+'.join(f'1|za{j}_{m}' for j in range(nf+1))
        L.append(f'        pr = {{ z_{m}*{nf+1} -> {za_t} }};')
        L.append('')
        for i in range(nf):
            xv = xi[i]
            if abs(xv) < 1e-9:
                L.append(f'        pr = {{ 0.000001*za{i}_{m}-0.000001 -> 1|gw{i}_{m} }};')
            else:
                L.append(f'        pr = {{ {f(xv)}*za{i}_{m}-{f(xv*yi)} -> 1|gw{i}_{m} }};')
        L.append(f'        pr = {{ za{nf}_{m}-{f(yi)} -> 1|gb_{m} }};')
        L.append('')
        L.append(f"        var0 = ({','.join(['0']*((nf+1)+1+(nf+1)+nf+1))});")
        L += [f'    }};', '']

    L.append('}')
    with open(fname, 'w') as fh:
        fh.write('\n'.join(L))
    return fname

def extract_weights(output_text, nf):
    """Parse w0..wN-1 and b from bnps3.py stdout."""
    w_vals = []
    for i in range(nf):
        m = re.search(rf'(?:^|\s)w{i}\s*[=:]\s*([-+]?[\d.eE]+)', output_text, re.MULTILINE)
        if m: w_vals.append(float(m.group(1)))
    mb   = re.search(r'(?:^|\s)b\s*[=:]\s*([-+]?[\d.eE]+)', output_text, re.MULTILINE)
    bias = float(mb.group(1)) if mb else 0.0
    return np.array(w_vals) if len(w_vals) == nf else None, bias

if __name__ == '__main__':
    print(f"Dataset loaded: {N_TRAIN} train / {N_TEST} test, {NF} features")
    print(f"MEMBRANE_COUNTS : {MEMBRANE_COUNTS}")
    print(f"STEPS_SWEEP     : {STEPS_SWEEP}")


Dataset loaded: 455 train / 114 test, 30 features
MEMBRANE_COUNTS : [25, 50, 100, 150, 200]
STEPS_SWEEP     : [10, 25, 50, 100]
